In [ ]:
# --- bootstrap: make src/ importable and run from the repository root ---
import os, sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "path_info.py").exists())
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

In [ ]:
# !pip install adjustText

In [ ]:
# tested with
# scikit-learn==1.5.0
# captum==0.7.0

In [ ]:
# !pip install statsmodels

In [ ]:
#code for autoreload script associated with jupyter notebook
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
import torch.nn.functional as F
from torchvision import transforms, datasets
from transformers import BlipModel, BlipProcessor
from tqdm import tqdm
import pandas as pd
import os
from sklearn.model_selection import StratifiedKFold
import numpy as np
import glob
from PIL import Image
from sklearn.metrics import f1_score, accuracy_score, classification_report
import re
from functools import lru_cache
import multiprocessing
import mmap
import json

from transformers import BlipModel, BlipProcessor, CLIPModel, CLIPProcessor

from baseline_model import CustomClassifier, plot_history
from CBM_model import CBM_model, plot_concept_to_class_weights

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="huggingface_hub.file_download")

os.environ["TOKENIZERS_PARALLELISM"] = "false" # to remove warnings about parallelism :
# huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
# To disable this warning, you can either:
# - Avoid using `tokenizers` before the fork if possible
# - Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)

In [ ]:
from CBM_pipeline import batch_size, device, num_workers, max_len

In [ ]:
from path_info import PATH

# the paper reports results for both backbones: 'clip' and 'clip-large'
backbone = 'clip'
# the concept_level values below were selected with 'clip' (minimum level reaching 99% coverage)

# N24

In [ ]:
from CBM_pipeline import run_CBM
concept_scores = f'{PATH}/datasets/N24News/combine/cb_llm_annotation/{backbone}_multimodal/outputs_concept_scoring/blue_checkpoints/{backbone}/cavs/regression/sorted_macro_concepts_coverage_MJ_cb_llm_abs_all_LIG.pkl'

In [ ]:
# choose minimum concept level so that coverage is at least 99%

model = run_CBM(dataset='N24', 
                dataset_type='CBLLM', 
                combine_type='combine', 
                backbone=backbone, 
                concept_representation='importance', 
                num_epochs=10, 
                import_concept_list=concept_scores,
                concept_level=12,
                load=False, 
                plot=True, 
                leakage_loss=True, 
                leakage_loss_activation='up', 
                kan_layer=True)

# CUB

In [ ]:
from CBM_pipeline import run_CBM
concept_scores = f'{PATH}/datasets/CUB_200_2011/combine/cb_llm_annotation/{backbone}_image/outputs_concept_scoring/blue_checkpoints/{backbone}/cavs/regression/sorted_macro_concepts_coverage_MJ_cb_llm_abs_all_LIG.pkl'

In [ ]:
# choose minimum concept level so that coverage is at least 99%

model = run_CBM(dataset='CUB', dataset_type='CBLLM', combine_type='combine', backbone=backbone, concept_representation='importance', num_epochs=10, import_concept_list=concept_scores, concept_level=0, load=False, plot=True, leakage_loss=True, leakage_loss_activation='up', kan_layer=True)

In [ ]:
# choose minimum concept level so that coverage is at least 99%

# Same but with cos cubed loss, should improve performance on CUB

model = run_CBM(dataset='CUB', 
                dataset_type='CBLLM', 
                combine_type='combine', 
                backbone=backbone, 
                concept_representation='importance', 
                num_epochs=10, 
                import_concept_list=concept_scores, 
                concept_level=0, 
                load=False, 
                plot=True, 
                leakage_loss=True, 
                leakage_loss_activation='up', 
                kan_layer=True, 
                loss_CBLLM='cos_cubed')

In [ ]:
# Take all CUB concepts to cross check performance

# Same but with cos cubed loss, should improve performance on CUB

model = run_CBM(dataset='CUB', 
                dataset_type='CBLLM', 
                combine_type='combine', 
                backbone=backbone, 
                concept_representation='importance', 
                num_epochs=10, 
                load=False, 
                plot=True, 
                leakage_loss=True, 
                leakage_loss_activation='up', 
                kan_layer=True, 
                loss_CBLLM='cos_cubed')

In [ ]:
# Take all CUB concepts to cross check performance

# Same but with cos cubed loss, should improve performance on CUB

# no KAN

model = run_CBM(dataset='CUB', 
                dataset_type='CBLLM', 
                combine_type='combine', 
                backbone=backbone, 
                concept_representation='importance', 
                num_epochs=10, 
                load=False, 
                plot=True, 
                leakage_loss=True, 
                leakage_loss_activation='up', 
                kan_layer=False, 
                loss_CBLLM='cos_cubed')

# agnews

In [ ]:
from CBM_pipeline import run_CBM
concept_scores = f'{PATH}/datasets/agnews/text/cb_llm_annotation/{backbone}_text/outputs_concept_scoring/blue_checkpoints/{backbone}/cavs/regression/sorted_macro_concepts_coverage_MJ_cb_llm_abs_all_LIG.pkl'

In [ ]:
# choose minimum concept level so that coverage is at least 99%

model = run_CBM(dataset='agnews', 
                dataset_type='CBLLM', 
                combine_type='text', 
                backbone=backbone, 
                concept_representation='importance', 
                num_epochs=10, 
                import_concept_list=concept_scores, 
                concept_level=7, 
                load=False, 
                plot=True, 
                leakage_loss=True, 
                leakage_loss_activation='up', 
                kan_layer=True)

# dbpedia

In [ ]:
from CBM_pipeline import run_CBM
concept_scores = f'{PATH}/datasets/dbpedia/text/cb_llm_annotation/{backbone}_text/outputs_concept_scoring/blue_checkpoints/{backbone}/cavs/regression/sorted_macro_concepts_coverage_MJ_cb_llm_abs_all_LIG.pkl'

In [ ]:
# choose minimum concept level so that coverage is at least 99%

model = run_CBM(dataset='dbpedia', 
                dataset_type='CBLLM', 
                combine_type='text', 
                backbone=backbone, 
                concept_representation='importance', 
                num_epochs=10, 
                import_concept_list=concept_scores, 
                concept_level=5, 
                load=False, 
                plot=True, 
                leakage_loss=True, 
                leakage_loss_activation='up', 
                kan_layer=True)